# DIVRS — Reproduce trên Colab (ML-10M)

Code đã fix sẵn ở repo **[HatDuaa/DIVRS-reproduce](https://github.com/HatDuaa/DIVRS-reproduce)**. Notebook: **clone repo → tải data → chạy**.

⚡ **Nên bật GPU**: Runtime → Change runtime type → **T4 GPU** (cell chạy tự dò GPU/CPU).

## 1. GPU + lib

In [ ]:
!nvidia-smi || echo 'KHONG co GPU -> chay CPU (cham)'
!nproc
!pip install -q absl-py setproctitle deprecated faiss-cpu
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Lấy code (hard reset về bản mới nhất trên GitHub)
Dòng `git log` cuối in commit đang dùng — phải là commit mới nhất của repo.

In [ ]:
import os
REPO = '/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
!git log --oneline -1

## 3. Tải data ML-10M (đã tiền xử lý, format DICE)
Tải `ml10m.zip` (~42 MB) → giải nén ra `data/ml10m/output/` (đủ 6 file bắt buộc + 3 graph).

In [ ]:
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip

need = ['train_coo_record.npz','val_coo_record.npz','test_coo_record.npz',
        'train_skew_coo_record.npz','popularity.npy','popularity_blend.npy']
miss = [f for f in need if not os.path.exists('data/ml10m/output/'+f)]
print('Data thieu:', miss if miss else 'KHONG — du, san sang!')

## 4. Train + Test — CHẠY THỬ (vòng train nhỏ)
Bản smoke-test: `epochs=3` cho nhanh, chỉ để thấy trọn luồng train→test→Recall (số sẽ thấp vì train ít). Muốn số đẹp đối chiếu paper thì tăng epochs (vd 30–50) hoặc bỏ flag (mặc định 500 + early-stop).

In [ ]:
import torch
USE_GPU = torch.cuda.is_available()
print('use_gpu =', USE_GPU)
!python app.py \
  --flagfile config/ml10m_DIVRS.cfg \
  --output ./output/ \
  --use_gpu={USE_GPU} --gpu_id=0 \
  --cg_use_gpu=False \
  --num_workers=2 \
  --epochs=3 --es_patience=2

## 5. Ghi chú
- **Chậm là do negative sampler** (quét mảng popularity mỗi sample) — CPU-bound, GPU không cứu được. Giảm `epochs` là cách nhanh nhất để chạy thử.
- **Baseline so sánh**: đổi `--flagfile` → `config/ml10m_mf.cfg` / `ml10m_ips.cfg` / `ml10m_cause.cfg` / `ml10m_lgn.cfg`.
- **GCN** (`config/ml10m_DIVRS-GCN.cfg`): cần cài `dgl` khớp torch; file `*_coo_adj_graph.npz` đã có sẵn.
- Kết quả Recall@K/NDCG in ra log; lưu ở `./output/`. Sửa code thì commit vào repo rồi cell 2 tự reset về bản mới.